# 02 — Model Schema Discovery & Profiler

Zero-assumption schema profiler that discovers entity structures across arbitrary provider formats (OpenAI `/v1`, Anthropic, Google Gemini `v1beta`, custom specs). Identifies the most complete archetype model and maps nested provider fields to standardized table columns.

## 1. Select Dataset Snapshot

In [ ]:
dataset_file = 'featherless-models-083126.json'

## 2. Universal Schema Discovery & Archetype Inspection

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd

# 1. Resolve Target File
target_name = globals().get("dataset_file", "").strip() or "featherless-models-083126.json"

possible_paths = [
    Path(target_name),
    Path("datasets") / target_name,
    Path("deepnote/datasets") / target_name,
    Path("/work/datasets") / target_name,
    Path("/workspaces/model-config-crafter/datasets") / target_name
]

resolved_file = None
for p in possible_paths:
    if p.exists() and p.is_file():
        resolved_file = p
        break

if not resolved_file:
    raise FileNotFoundError(f"[FAIL FAST] Could not locate dataset snapshot '{target_name}'. Checked: {[str(p) for p in possible_paths]}")

print(f"[Schema Profiler] Loading dataset from: {resolved_file}")
with open(resolved_file, "r", encoding="utf-8") as f:
    raw_payload = json.load(f)

# 2. Agnostic Entity Collection Extraction
def extract_entities(raw_json):
    if isinstance(raw_json, list):
        items = [x for x in raw_json if isinstance(x, dict)]
        if items:
            return items, "<root:list>"
    elif isinstance(raw_json, dict):
        best_list, best_path = [], ""
        def search(d, current=""):
            nonlocal best_list, best_path
            for k, v in d.items():
                p = f"{current}.{k}" if current else k
                if isinstance(v, list):
                    dicts = [x for x in v if isinstance(x, dict)]
                    if len(dicts) > len(best_list):
                        best_list, best_path = dicts, p
                elif isinstance(v, dict):
                    vals = [x for x in v.values() if isinstance(x, dict)]
                    if len(vals) > len(best_list) and len(vals) > 1:
                        best_list, best_path = vals, f"{p}.<values>"
                    else:
                        search(v, p)
        search(raw_json)
        if best_list:
            return best_list, best_path
    raise ValueError("Could not automatically locate model entities array in JSON payload.")

entities, envelope_location = extract_entities(raw_payload)

# 3. Recursive Path Extraction
def extract_paths(obj, prefix=""):
    paths = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            if v is not None and v != "" and v != [] and v != {}:
                paths[p] = {"type": type(v).__name__, "val": v if not isinstance(v, (dict, list)) else f"<{type(v).__name__}>"}
                if isinstance(v, (dict, list)):
                    paths.update(extract_paths(v, p))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            p = f"{prefix}[{i}]"
            if item is not None and item != "":
                paths[p] = {"type": type(item).__name__, "val": item if not isinstance(item, (dict, list)) else f"<{type(item).__name__}>"}
                if isinstance(item, (dict, list)):
                    paths.update(extract_paths(item, p))
    return paths

# 4. Profile All Entities & Build Superset Schema
global_counts = {}
global_types = {}
scored = []

for idx, e in enumerate(entities):
    f_map = extract_paths(e)
    # Infer entity ID
    m_id = e.get("id") or e.get("name") or e.get("model_id") or e.get("slug") or f"model_{idx}"
    for path, meta in f_map.items():
        global_counts[path] = global_counts.get(path, 0) + 1
        if path not in global_types:
            global_types[path] = set()
        global_types[path].add(meta["type"])
    scored.append({"id": str(m_id), "score": len(f_map), "fields": f_map, "raw": e, "size": len(json.dumps(e))})

scored.sort(key=lambda x: (x["score"], x["size"]), reverse=True)
top_candidate = scored[0]

print(f"\n=======================================================")
print(f" DATASET SCHEMA SUMMARY: {resolved_file.name}")
print(f"=======================================================")
print(f"• Discovered Envelope Location : {envelope_location}")
print(f"• Total Models in Dataset       : {len(entities)}")
print(f"• Global Superset Schema        : {len(global_counts)} unique paths")
print(f"• Top Model Coverage Score      : {top_candidate['score']}/{len(global_counts)} paths ({round(top_candidate['score']/len(global_counts)*100, 1)}%)")
print(f"• Model Archetype Candidate     : '{top_candidate['id']}'")
print(f"=======================================================\n")

# Display Field Distribution Table
freq_data = []
for p, count in sorted(global_counts.items(), key=lambda x: x[1], reverse=True):
    freq_data.append({
        "Field Path": p,
        "Type": "|".join(global_types[p]),
        "Populated Count": count,
        "Coverage %": f"{round((count / len(entities)) * 100, 1)}%"
    })

freq_df = pd.DataFrame(freq_data)
print("Discovered Field Distribution (Top 25 most common):")
display(freq_df.head(25))

print(f"\nArchetype Model Payload Preview for '{top_candidate['id']}':")
print(json.dumps(top_candidate["raw"], indent=2))


## 3. Configure Column Mappings & Fallback Overrides

Specify which discovered JSON dot-paths map to standardized table columns. If a model lacks a context window, the fallback value will be used.

In [ ]:
model_id_field = 'id'

In [ ]:
context_length_field = 'context_length'

In [ ]:
input_cost_field = 'pricing.input'

In [ ]:
output_cost_field = 'pricing.output'

In [ ]:
default_context_length = '32768'

## 4. Normalized Dataframe Extraction & Preview

In [ ]:
# 1. Resolve Configured Field Paths
m_id_key = globals().get("model_id_field", "").strip() or "id"
ctx_key = globals().get("context_length_field", "").strip()
in_cost_key = globals().get("input_cost_field", "").strip()
out_cost_key = globals().get("output_cost_field", "").strip()

try:
    fallback_ctx = int(str(globals().get("default_context_length", "32768")).strip())
except ValueError:
    fallback_ctx = 32768

def get_nested_val(obj, dot_path):
    if not dot_path:
        return None
    parts = dot_path.split(".")
    curr = obj
    for p in parts:
        if isinstance(curr, dict) and p in curr:
            curr = curr[p]
        else:
            return None
    return curr

# 2. Extract Normalized Records
normalized_rows = []
for idx, entity in enumerate(entities):
    # Model ID
    m_id = get_nested_val(entity, m_id_key) or entity.get("id") or entity.get("name") or f"model_{idx}"
    
    # Context Length with Fallback Override
    ctx = get_nested_val(entity, ctx_key) if ctx_key else None
    if ctx is None or ctx == "":
        ctx = fallback_ctx
    else:
        try:
            ctx = int(ctx)
        except (ValueError, TypeError):
            ctx = fallback_ctx

    # Pricing (Input / Output)
    in_cost = get_nested_val(entity, in_cost_key) if in_cost_key else None
    out_cost = get_nested_val(entity, out_cost_key) if out_cost_key else None

    # Capabilities heuristics
    tools_val = get_nested_val(entity, "features.tool_use")
    if tools_val is None:
        supp_params = entity.get("supported_parameters", [])
        tools_val = "tools" in supp_params if isinstance(supp_params, list) else None

    normalized_rows.append({
        "id": str(m_id),
        "context_length": ctx,
        "pricing_input": in_cost,
        "pricing_output": out_cost,
        "tools_supported": tools_val,
        "_raw": entity
    })

df_normalized = pd.DataFrame(normalized_rows)

print(f"[Schema Extraction] Successfully normalized {len(df_normalized)} models.")
print(f"• ID Column: '{m_id_key}'")
print(f"• Context Length: '{ctx_key or 'None (using fallback)'}' (Fallback: {fallback_ctx})")
print(f"• Input Pricing: '{in_cost_key or 'None'}'")
print(f"• Output Pricing: '{out_cost_key or 'None'}'")
print("\nFirst 10 Normalized Records:")
display(df_normalized[["id", "context_length", "pricing_input", "pricing_output", "tools_supported"]].head(10))
